<a href="https://colab.research.google.com/github/ljzier/ST-554-repo/blob/main/Zier_ST_554_HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Linda Zier**

**ST 554**

**HW #7**

**Goal**

The purpose of this homework is to practice fitting MLR and logistic regression models (including penalized
or regularized models).

**Data**

We will use a dataset from the UCI Machine Learning Repository. This data set is about wine quality and includes the following Input variables:

        1 - fixed acidity
        2 - volatile acidity
        3 - citric acid
        4 - residual sugar
        5 - chlorides
        6 - free sulfur dioxide
        7 - total sulfur dioxide
        8 - density
        9 - pH
        10 - sulphates
        11 - alcohol **
        12 - quality
        


*   Rather than try to predict quality, *alcohol* will be our target variable.

*   For fitting logistic regression type models we’ll use the *type of wine* as the response variable.

So let's get started!

# Train Models

In [184]:
# imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, \
log_loss, accuracy_score
from sklearn.linear_model import LinearRegression, LassoCV, Lasso, RidgeCV, \
Ridge, ElasticNetCV, ElasticNet, LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import PolynomialFeatures

!git clone https://github.com/ljzier/ST-554-repo.git

red_wine = pd.read_csv('ST-554-repo/data/winequality-red.csv', sep=';')
white_wine = pd.read_csv('ST-554-repo/data/winequality-white.csv', sep=';')


I've assigned a new variable so I can combine the red and white into one dataset.


In [ ]:
# assign new variable
red_wine['type'] = 0
white_wine['type'] = 1

#combine
wine = pd.concat([red_wine, white_wine])

#checking for null values. there is none
wine.isnull().sum()

wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0


I split the data into training and test sets being sure there were similar portions of red and white wine in each. My X is the wine data minus the alcohol column. My response, y, is the alcohol.

Next I standardized the data.


In [ ]:
#split data into training and text set including
# equal amounts of red and white wine

X_train, X_test, y_train, y_test = train_test_split(
  wine.drop("alcohol", axis = 1),
  wine["alcohol"],
  test_size=0.20,
  random_state=41, shuffle = True,
  stratify=wine['type'])

# I'm using means and stds like in class, but
means = X_train.apply(np.mean, axis = 0)
stds = X_train.apply(np.std, axis = 0)

X_train = X_train.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
2586,-0.324228,-0.966444,-0.127270,1.536375,0.161139,-0.644615,0.993523,1.114684,-1.362938,-0.075387,1.351979,0.571351
2000,-0.324228,-0.179618,-1.584610,-0.853833,-0.452065,1.443989,0.529082,-0.831770,-0.176317,-0.474290,-0.929266,0.571351
3032,-0.401439,-1.208544,0.983084,-0.811899,-0.563556,-0.870410,-0.417662,-0.686117,0.947850,-0.274839,-0.929266,0.571351
2410,-0.169806,-1.087494,-0.404859,0.383205,-0.256954,-0.023679,1.868815,0.498969,1.010303,0.323515,0.211357,0.571351
1195,0.525093,-0.724344,0.913687,1.829910,-0.507811,1.274643,0.457630,1.485437,-1.175577,-0.474290,0.211357,0.571351
...,...,...,...,...,...,...,...,...,...,...,...,...
4300,-0.633072,-0.300669,-0.127270,0.215471,4.258459,1.782681,0.725577,0.022286,-0.738401,-0.873192,-0.929266,0.571351
314,-1.096338,0.001957,-0.751845,0.236438,-0.452065,1.105296,0.922071,-0.202814,0.448220,0.589450,-0.929266,0.571351
1526,-0.324228,0.788782,-1.654007,-0.686099,0.216885,-0.701064,-1.382270,0.270559,0.510674,0.788901,0.211357,-1.750237
376,-0.633072,-0.179618,-0.266065,-0.832866,-0.535684,-1.039756,0.064642,-0.931079,0.635581,-0.141871,1.351979,0.571351


##MLR Models
 Below I fit different multiple linear regression models. I used the full model to decide which coefficients to include in the other models , discounting those that are closer to 0.


In [ ]:
# Full model
cv_full_model = cross_validate(
    LinearRegression(),
    X_train,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

# I looked at the coefficients so I could decide which to keep for my other
# models. Also I know that denser wines have less alcohol .
mlr_full = LinearRegression().fit(X_train, y_train)
print(mlr_full.intercept_)
print(np.array(list(zip(X_train.columns, mlr_full.coef_))))

# model with just the larger Betas in the full model
cv_mlr1 = cross_validate(
    LinearRegression(),
    X_train[["fixed acidity","residual sugar", "density"]],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

# model with the smaller Beta1s removed
cv_mlr2 = cross_validate(
    LinearRegression(),
    X_train.drop(["citric acid","chlorides", "free sulfur dioxide",
                  "total sulfur dioxide"], axis=1),
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

# creating interaction terms for training set
X_interaction = X_train[["density", "residual sugar", "type"]].copy()
X_interaction["sugar_x_type"] = X_train["residual sugar"] * X_train["type"]

#model with interaction term
cv_interaction = cross_validate(
    LinearRegression(),
    X_interaction,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error")

# creating polynomial term for training set
X_poly = X_train[["density", "residual sugar", "fixed acidity"]].copy()
X_poly["density^2"]=X_train["density"]**2

# model with polynomial term
cv_poly = cross_validate(
    LinearRegression(),
    X_poly,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error")

# creating polynomial term for training set with smaller beta1s removed
X_train_poly2 =X_train.drop(["citric acid","chlorides", "free sulfur dioxide",
                       "total sulfur dioxide"], axis=1).copy()
X_train_poly2["density^2"] = X_train["density"]**2


# model with the smaller Beta1s removed and a squared density
cv_poly2 = cross_validate(
    LinearRegression(),
    X_train_poly2,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

10.489225835417813
[['fixed acidity' '0.6638593274119095']
 ['volatile acidity' '0.13530220234855728']
 ['citric acid' '0.06977401198189065']
 ['residual sugar' '1.087792121209095']
 ['chlorides' '-0.033421706437854096']
 ['free sulfur dioxide' '-0.06271149553763086']
 ['total sulfur dioxide' '-0.0177250744286494']
 ['density' '-1.9634150766059468']
 ['pH' '0.4164067544658201']
 ['sulphates' '0.14673473729505374']
 ['quality' '0.0921800902050636']
 ['type' '-0.4811188927095919']]


The model with the small Beta1s (from the full model) removed and a squared density is the best.

In [ ]:
print(np.sqrt(-sum(cv_full_model['test_score'])),
      np.sqrt(-sum(cv_mlr1['test_score'])),
      np.sqrt(-sum(cv_mlr2['test_score'])),
      np.sqrt(-sum(cv_interaction['test_score'])),
      np.sqrt(-sum(cv_poly['test_score'])),
      np.sqrt(-sum(cv_poly2['test_score'])))

1.1554751418867872 1.7782314722070856 1.1661597649018451 1.6071549129757186 1.7577207256837977 1.0085255699223084


Here is the best linear regression model.

In [ ]:
mlr_best = LinearRegression().fit(X_train_poly2, y_train)
print(mlr_best.intercept_)
print(np.array(list(zip(X_train_poly2.columns, mlr_best.coef_))))

10.409612446729302
[['fixed acidity' '0.7062592772105039']
 ['volatile acidity' '0.07656598422511207']
 ['residual sugar' '1.0766128269016515']
 ['density' '-2.051450998816228']
 ['pH' '0.41567934246055616']
 ['sulphates' '0.1422735099878461']
 ['quality' '0.054631349212210925']
 ['type' '-0.5452433629981144']
 ['density^2' '0.07961338868850773']]


## Lasso model
 I fit a lasso model with the full set of predictors and decided to let the methods decide what to keep. I then looked at tuning parameters and CV errors.

In [ ]:
# lasso model with full training set
lasso_mod = LassoCV(cv=5, random_state=0).fit(X_train,y_train)

# this is to look at tuning parms (alphas)
np.set_printoptions(suppress = True)
fit_info = np.array(list(zip(lasso_mod.alphas_, np.mean(lasso_mod.mse_path_, axis = 1))))
fit_info[fit_info[:,1].argsort()][0:10,].round(4)


array([[0.0015, 0.2669],
       [0.0014, 0.2669],
       [0.0013, 0.2669],
       [0.0012, 0.2669],
       [0.0016, 0.2669],
       [0.0012, 0.2669],
       [0.0011, 0.2669],
       [0.001 , 0.2669],
       [0.0009, 0.2669],
       [0.0009, 0.2669]])

This is the best Lasso model for comparison.

In [ ]:
lasso_best = Lasso(lasso_mod.alpha_).fit(X_train, y_train)
print(lasso_best.intercept_)
print(np.array(list(zip(X_train.columns, lasso_best.coef_))))

10.489225835417813
[['fixed acidity' '0.6535183328835275']
 ['volatile acidity' '0.13322346911356142']
 ['citric acid' '0.06761053595334136']
 ['residual sugar' '1.0663802875574073']
 ['chlorides' '-0.03284133019680471']
 ['free sulfur dioxide' '-0.060233044935596794']
 ['total sulfur dioxide' '-0.021365830954549164']
 ['density' '-1.938948743174578']
 ['pH' '0.40929495807700894']
 ['sulphates' '0.14396789565591844']
 ['quality' '0.09489503690086737']
 ['type' '-0.4709590730085519']]


## Ridge Regression

We'll repeat the previous with the full model again.

In [ ]:
# Ridge model with full training set
ridge_mod = RidgeCV(cv=5).fit(X_train, y_train)

# the selected alpha
print(ridge_mod.alpha_)


10.0


This is the best Ridge model.

In [ ]:
ridge_best = Ridge(ridge_mod.alpha_).fit(X_train, y_train)
print(ridge_best.intercept_)
print(np.array(list(zip(X_train.columns, ridge_best.coef_))))

10.489225835417814
[['fixed acidity' '0.6454001832272651']
 ['volatile acidity' '0.1377605263164458']
 ['citric acid' '0.0715730211673687']
 ['residual sugar' '1.0534359178907104']
 ['chlorides' '-0.0380949845108338']
 ['free sulfur dioxide' '-0.05975226015036659']
 ['total sulfur dioxide' '-0.0273073708034198']
 ['density' '-1.9207109810245642']
 ['pH' '0.4061401456237321']
 ['sulphates' '0.14483204407293976']
 ['quality' '0.09914985178727867']
 ['type' '-0.4616225035600473']]


## Elastic Net
Finally, we'll do the elastic net with the full set of predictors

In [ ]:
# enet with full training set. varying the ratio of lasso to Ridge
enet_mod = ElasticNetCV(cv=5, random_state=0,l1_ratio=[0.1, 0.25, 0.5, 0.75, 0.9, 1.0]).fit(X_train, y_train)

# best tuning parameters
print(enet_mod.alpha_)
print(enet_mod.l1_ratio_)


0.0015357202781697654
1.0


This is the best Elastic Net model.

In [ ]:
# best elastic net
enet_best = ElasticNet(alpha=enet_mod.alpha_,l1_ratio=enet_mod.l1_ratio_).fit(X_train, y_train)
print(enet_best.intercept_)
print(np.array(list(zip(X_train.columns, enet_best.coef_))))

10.489225835417813
[['fixed acidity' '0.6535183328835275']
 ['volatile acidity' '0.13322346911356142']
 ['citric acid' '0.06761053595334136']
 ['residual sugar' '1.0663802875574073']
 ['chlorides' '-0.03284133019680471']
 ['free sulfur dioxide' '-0.060233044935596794']
 ['total sulfur dioxide' '-0.021365830954549164']
 ['density' '-1.938948743174578']
 ['pH' '0.40929495807700894']
 ['sulphates' '0.14396789565591844']
 ['quality' '0.09489503690086737']
 ['type' '-0.4709590730085519']]


#Testing Models

Now we'll compare the performance usng the test sets. They will all be using the same test set except the MLR model.


In [ ]:
#stanardizing the test sets using the professors method instead of StandardScalor
#quick function to standardize based off of a supplied mean and std

def my_std_fun(x, means, stds):
    return(x-means)/stds

#loop through the columns and use the function on each
for x in X_test.columns:
    X_test[x] = my_std_fun(X_test[x], means[x], stds[x])

X_test.head()

#rebuild X_test_poly2 off of that
X_test_poly2 =X_test.drop(["citric acid","chlorides", "free sulfur dioxide",
                       "total sulfur dioxide"], axis=1).copy()
X_test_poly2["density^2"] = X_test["density"]**2
X_test_poly2.head()

#verifying X_test is unchanged
X_test.head()
X_test.describe()

# predictions from each of the models
mlr_pred   = mlr_best.predict(X_test_poly2)
lasso_pred = lasso_best.predict(X_test)
ridge_pred = ridge_best.predict(X_test)
enet_pred  = enet_best.predict(X_test)

Comparing them all, my MLR model edged out the others by a little bit.  The other models are all similar with the LASSO and Elastic Net being identical since the ratio was 1.0 between Lasso and Ridge.

In [ ]:

models = {'MLR': mlr_pred, 'LASSO': lasso_pred, 'Ridge': ridge_pred, 'Elastic Net': enet_pred}

for name, pred in models.items():
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae  = mean_absolute_error(y_test, pred)
    print(f"{name}: RMSE = {rmse:.4f}, MAE = {mae:.4f}")

MLR: RMSE = 0.4640, MAE = 0.3441
LASSO: RMSE = 0.4715, MAE = 0.3563
Ridge: RMSE = 0.4721, MAE = 0.3574
Elastic Net: RMSE = 0.4715, MAE = 0.3563


In [ ]:
mlr_best = LinearRegression().fit(X_train_poly2, y_train)

print(mlr_best.intercept_)
print(np.array(list(zip(X_train_poly2.columns, mlr_best.coef_))))

10.409612446729302
[['fixed acidity' '0.7062592772105039']
 ['volatile acidity' '0.07656598422511207']
 ['residual sugar' '1.0766128269016515']
 ['density' '-2.051450998816228']
 ['pH' '0.41567934246055616']
 ['sulphates' '0.1422735099878461']
 ['quality' '0.054631349212210925']
 ['type' '-0.5452433629981144']
 ['density^2' '0.07961338868850773']]


Best model:

$$\hat{\text{alcohol}} = 10.410 + 0.706x_{\text{fixed acidity}} + 0.077x_{\text{volatile acidity}} + 1.077x_{\text{residual sugar}} - 2.051x_{\text{density}} + 0.416x_{\text{pH}} + 0.142x_{\text{sulphates}} + 0.055x_{\text{quality}} - 0.545x_{\text{type}} + 0.080x_{\text{density}^2}$$

# Classification Task
Red and White type is now the response variable.  I'm going to repeat the training and testing done previously but use logistic regression models since *type* is a categorical variable.

• log-loss or negative log-loss as the metric for choosing models during the training process

• During the testing portion, models will be compared on both log-loss and accuracy

I begain by splitting my data into training and test sets. I standardized the training set and then used the means and stds from that to also standardize the test set.

In [157]:
#split data into training and text sets

X_trainC, X_testC, y_trainC, y_testC = train_test_split(
  wine.drop("type", axis = 1),
  wine["type"],
  test_size=0.20,
  random_state=41,
  shuffle = True,
  stratify=wine['type'])

# I'm using means and stds like in class, but could use .Standard
meansC = X_trainC.apply(np.mean, axis = 0)
stdsC = X_trainC.apply(np.std, axis = 0)

# standardizing training set
X_trainC = X_trainC.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)



##Logistic Regression Models

In [165]:
# full model
cv_fullC = cross_validate(
    LogisticRegression(),
    X_trainC,
    y_trainC,
    cv=5,
    scoring="neg_log_loss")

#reduced model
cv_reducedC = cross_validate(
    LogisticRegression(),
    X_trainC[["residual sugar","total sulfur dioxide", "density", "volatile acidity"]],
    y_trainC,
    cv=5,
    scoring="neg_log_loss")

# model with interaction terms
X_interactionC = X_trainC[["residual sugar","total sulfur dioxide", "density", "volatile acidity"]].copy()
X_interactionC["sugar_x_total"] = X_trainC["residual sugar"] * X_trainC["total sulfur dioxide"]

cv_interactionC = cross_validate(
    LogisticRegression(),
    X_interactionC,
    y_trainC,
    cv=5,
    scoring="neg_log_loss")

#model with polynomial term
X_polyC = X_trainC[["residual sugar","total sulfur dioxide", "density", "volatile acidity", "pH"]].copy()
X_polyC["sugar_x_pH"] = X_trainC["residual sugar"] * X_trainC["pH"]
X_polyC["density^2"] = X_trainC["density"]**2

cv_polyC = cross_validate(
    LogisticRegression(),
    X_polyC,
    y_trainC,
    cv=5,
    scoring="neg_log_loss")


We can judge these by comparing the mean CV log-loss across these models. In this case the full model is best because it is closest to 0.

In [191]:
print(np.mean(cv_fullC['test_score']),
      np.mean(cv_reducedC['test_score']),
      np.mean(cv_interactionC['test_score']),
      np.mean(cv_polyC['test_score']))


-0.03944214149315208 -0.05440340007409498 -0.05518627958390469 -0.05386147316490088


##Penalization Methods with Logistic Regression

To do Lasso, Ridge, and Elastic Net, the penalty setting in the Logistic Regression call gets changed.

###Lasso
The penalty C, is the inverse of the regularization strength so larger means less penalty. (0.359 is moderate).  It zeroed out fixed acidity and pH.

In [180]:
# Lasso logistic regression
Lasso_logit = LogisticRegressionCV(cv=5, penalty='l1', solver='liblinear',
                                   scoring='neg_log_loss', random_state=0).fit(X_trainC, y_trainC)

print(Lasso_logit.C_)
print(Lasso_logit.intercept_)
print(np.array(list(zip(X_trainC.columns, Lasso_logit.coef_[0]))))

[0.35938137]
[3.81155809]
[['fixed acidity' '0.0']
 ['volatile acidity' '-1.2164095401143842']
 ['citric acid' '0.2789479449606744']
 ['residual sugar' '3.4389752517911183']
 ['chlorides' '-0.7515387244203713']
 ['free sulfur dioxide' '-0.7065234162777093']
 ['total sulfur dioxide' '2.7216081321724985']
 ['density' '-3.884482588791108']
 ['pH' '0.0']
 ['sulphates' '-0.6548192728609676']
 ['alcohol' '-1.2418744651180542']
 ['quality' '-0.34764015677384197']]


###Ridge
Ridge has the same C value and a similar intercept.  No values were zeroed out.

In [182]:
# Ridge logistic regression
Ridge_logit = LogisticRegressionCV(cv=5, penalty='l2',
                                   scoring='neg_log_loss').fit(X_trainC, y_trainC)

print(Ridge_logit.C_)
print(Ridge_logit.intercept_)
print(np.array(list(zip(X_trainC.columns, Ridge_logit.coef_[0]))))

[0.35938137]
[3.82621817]
[['fixed acidity' '-0.7115437807659544']
 ['volatile acidity' '-1.3548468787956698']
 ['citric acid' '0.33637341076962957']
 ['residual sugar' '2.378516739470481']
 ['chlorides' '-0.904742024814014']
 ['free sulfur dioxide' '-0.49503717215640797']
 ['total sulfur dioxide' '2.6026834988295153']
 ['density' '-2.403792811823475']
 ['pH' '-0.5035645246725019']
 ['sulphates' '-0.8658664665441977']
 ['alcohol' '-0.6590504610877277']
 ['quality' '-0.2988551591431676']]


### Elastic Net
The elastic net model chose a ratio of 0.5 indicating the penalty was equal parts Lasso and Ridge. No parameters were zeroed out completely.

In [179]:
#Elastic Net regression
enet_logit = LogisticRegressionCV(cv=5, penalty='elasticnet', solver='saga',
                                      l1_ratios=[0.1, 0.25, 0.5, 0.75, 0.9, 1.0],
                                      scoring='neg_log_loss', max_iter=1000).fit(X_trainC, y_trainC)

print(enet_logit.C_)
print(enet_logit.l1_ratio_)
print(enet_logit.intercept_)
print(np.array(list(zip(X_trainC.columns, enet_logit.coef_[0]))))

[0.35938137]
[0.5]
[3.91686787]
[['fixed acidity' '-0.476022914855685']
 ['volatile acidity' '-1.339510882498203']
 ['citric acid' '0.29322049005078443']
 ['residual sugar' '2.783167518543436']
 ['chlorides' '-0.8599480902804388']
 ['free sulfur dioxide' '-0.5627544224209535']
 ['total sulfur dioxide' '2.675587590635147']
 ['density' '-2.860148968578575']
 ['pH' '-0.355599426319954']
 ['sulphates' '-0.8203303147966061']
 ['alcohol' '-0.8293839505634718']
 ['quality' '-0.30462983511516506']]


In [193]:

# get predictions
full_pred_prob = cv_fullC.predict_proba(X_testC)
lasso_pred_prob = lasso_logit.predict_proba(X_testC)
ridge_pred_prob = ridge_logit.predict_proba(X_testC)
enet_pred_prob = enet_logit.predict_proba(X_testC)

# log loss needs probabilities
print("Full log-loss:", log_loss(y_testC, full_pred_prob))
print("LASSO log-loss:", log_loss(y_testC, lasso_pred_prob))
print("Ridge log-loss:", log_loss(y_testC, ridge_pred_prob))
print("Elastic Net log-loss:", log_loss(y_testC, enet_pred_prob))

# accuracy needs class predictions
print("Full accuracy:", accuracy_score(y_testC, full_logit.predict(X_testC)))
print("LASSO accuracy:", accuracy_score(y_testC, lasso_logit.predict(X_testC)))
print("Ridge accuracy:", accuracy_score(y_testC, ridge_logit.predict(X_testC)))
print("Elastic Net accuracy:", accuracy_score(y_testC, enet_logit.predict(X_testC)))

NameError: name 'lasso_logit' is not defined